# Librerias a utilizar

In [ ]:
# Observar todas las librerias a usar y rellenar esta seccion


#  Observar el dataset CIFAR 10

In [ ]:
# Cargar el Dataset CIFAR 10
from keras.datasets import ___

# load dataset
(X_train, objeto), (_, _) = cifar10.load_data()    # Forma correcta?  ¿Porque?
print('Datos: X=%s, y=%s' % (X_train.shape, objeto.shape))

# plot first few images
for i in range(9):
	# define subplot
	plt.subplot(330 + 1 + i)
	# plot raw pixel data
	plt.imshow(X_train[i])
# Mostrar las imagenes
plt.___()

In [ ]:
# Seleccionar el tipo de dato a entrenar dentro de las clases
X_train_10 = []
for j in range(0,len(X_train)):
  if objeto[j] == 7:
    X_train_10.append(X_train[j])

print('Cantidad de objetos: ', ___(X_train_10) )

# plot first few images
for i in range(9):
	# define subplot
	plt.subplot(330 + 1 + i)
	# plot raw pixel data
	plt.imshow(X_train_10[i])
# show the figure
___.show()

# Generar el modelo de WGAN

In [ ]:
# Definir el modelo del variational autoencoder
input_size = (___,___,___)
latent_dim = 100
d_p = 0.2

#Discriminador / en este caso tambien llamada funcion critica

x = Input(input_size)
conv1 = Conv2D(64,5,strides=(2,2),padding='same',activation=LeakyReLU(alpha=0.2))(x)
conv2 = Conv2D(64,5,strides=(2,2),padding='same',activation=LeakyReLU(alpha=0.2))(conv1)
conv3 = Conv2D(128,5,strides=(2,2),padding='same',activation=LeakyReLU(alpha=0.2))(conv2)
conv4 = Conv2D(128,5,strides=(1,1),padding='same',activation=LeakyReLU(alpha=0.2))(conv3)
f = Flatten()(conv4)
d1 = Dense(1)(f) # Notar que ahora ya no tenemos activacion



#Generador
g_x = Input(shape=(latent_dim))
d = Dense(4096)(g_x)
d = BatchNormalization()(d)
r = Reshape((8,8,64))(d)

dec1 = UpSampling2D(size=(2,2))(r)
dec1 = Conv2D(128,5,strides=(1,1), padding='same', activation=LeakyReLU(alpha=0.2))(dec1)
dec1 = BatchNormalization(momentum=0.8)(dec1)

dec2 = UpSampling2D(size=(2,2))(dec1)
dec2 = Conv2D(64,5,strides=(1,1), padding='same', activation=LeakyReLU(alpha=0.2))(dec2)
dec2 = BatchNormalization(momentum=0.8)(dec2)

dec3 = Conv2D(64,5,strides=(1,1), padding='same', activation=LeakyReLU(alpha=0.2))(dec2)
dec3 = BatchNormalization(momentum=0.8)(dec3)

dec4 = Conv2D(3,5,strides=(1,1), padding='same', activation='tanh')(dec3)

In [ ]:
# Wasserstein Loss Function
def wasserstein_loss(y_true, y_pred):
	return K.mean(y_true * y_pred)

In [ ]:
# Compilar los modelos
opti = RMSprop(lr=0.00005)  # No utilizams ADAM
clip_value = 0.01
n_critic = 5 # Numero de entrenamiento de la funcion critica por veces del generador

# Definir discriminador y generador
discriminador = Model(x, d1, name= 'discriminador')
discriminador.compile(optimizer=opti, loss=___, metrics=[___])
discriminador.summary()

generador = Model(g_x, dec4, name='generador')
generador.summary()

# Modelo combinado
discriminador.trainable = False
sal_dis = discriminador(generador(g_x))
modelo_t = Model(g_x, sal_dis)
modelo_t.compile(optimizer=___, loss=___, ___=['accuracy'])
___.___()  # Mostrar el modelo

In [ ]:
# Se necesita entrenar en dos partes

# Primero el discriminador para que diferencia imagenes reales de ruido
# Con ayuda del discriminador se entrena el generador

def train(epochs, batch_size=128, sample_interval=50):#
        X_train= np.array(X_train_10)
        X_train = X_train / 127.5 - 1.
        X_train = np.clip(X_train, -1, 1)
        X_train = np.expand_dims(X_train, axis=-1)

        valid = -np.ones((batch_size, 1))   # Tomar los valores como reales
        fake = np.ones((batch_size, 1))   # Tomar los valores como falsos, anteriormente eran 0

        for epoch in range(epochs):

          discriminador.trainable = True
          for _ in range(n_critic):

            # ENTRENAR EL DSICRIMINADOR (CRITIC)

            idx = np.random.randint(0, X_train.shape[0], batch_size) # Escoje numero aleatoriamente
            imgs = X_train[idx]                                      # Toma idx imagenes reales

            noise = np.random.normal(0, 1, (batch_size, latent_dim)) # crea ruidos aleatorios
            
            gen_imgs = generador.predict(noise)                      # Crea objetos falsos

            d_loss_real = discriminador.train_on_batch(imgs, valid)  # Observa los objetos reales
            d_loss_fake = discriminador.train_on_batch(gen_imgs, fake) # Observa los objetos falsos
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)            # Junta las dos perdidas del discriminador

            # Recortar los valores de los pesos dentro de la funcion critica
            #---------------------------------------------------------------
            for l in discriminador.layers:
              weights = l.get_weights()
              weights = [np.clip(w, -clip_value, clip_value) for w in weights]
              l.set_weights(weights)
            #----------------------------------------------------------------

          # Entrenar el generador
          discriminador.trainable = False
          noise = np.random.normal(0, 1, (batch_size, latent_dim))
          g_loss = modelo_t.train_on_batch(noise, valid)           # Observa la perdida del modelo total

          print ('Epoca: ',epoch, '  D_loss: ', round(d_loss[0],3), '  G_loss: ', round(g_loss[0],3))
          if epoch % sample_interval == 0:
            sample_images(epoch)

def sample_images(epoch):
        r, c = 5, 5
        noise = np.random.normal(0, 1, (r * c, latent_dim))
        gen_imgs = generador.predict(noise)
        gen_imgs = 0.5 * gen_imgs + 0.5
        fig, axs = plt.subplots(r, c)
        cnt = 0
        for i in range(r):
            for j in range(c):
                axs[i, j].imshow(gen_imgs[cnt, :, :, :])
                axs[i, j].axis('off')
                cnt += 1
        plt.show()

In [ ]:
___(___=10000, batch_size=64, sample_interval=100)

In [ ]:
# probar el modelo terminado
sample_images(epoch=2000)